# Text Classification
## Text Classification with Representation Model
- Choosing the right model is a straightforward as you might think with over 60000 models on the: https://huggingface.co/models?pipeline_tag=text-classification
- RoBerta, DistilBERT, ALBERT, and DeBERTa
- https://huggingface.co/google-bert/bert-base-uncased
- https://huggingface.co/FacebookAI/roberta-base
- https://huggingface.co/distilbert/distilbert-base-uncased
- https://huggingface.co/microsoft/deberta-base
- https://huggingface.co/prajjwal1/bert-tiny
- https://huggingface.co/albert/albert-base-v2
- A place to select the most relevant embedding model: https://huggingface.co/spaces/mteb/leaderboard




### USing a task-Specific Model

In [ ]:
# pip install sentence-transformers

In [ ]:
# pip install -U datasets

In [3]:
from datasets import load_dataset

data = load_dataset("rotten_tomatoes")

d:\2026-courses\LLMs-Handson\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [120]:
data.column_names
# How many unique labels are there?
print("Unique labels:", set(data["test"]["label"]))


Unique labels: {0, 1}


In [5]:
print(data["train"])
print(data["test"])
print(data["validation"])

Dataset({
    features: ['text', 'label'],
    num_rows: 8530
})
Dataset({
    features: ['text', 'label'],
    num_rows: 1066
})
Dataset({
    features: ['text', 'label'],
    num_rows: 1066
})


In [ ]:
# get the first example  and last example
data["test"][0,-1]

{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
  'things really get weird , though not particularly scary : the movie is all portent and no content .'],
 'label': [1, 0]}

## Text Classification with Representation Models
### Using a Task-specific Model
- Task-specific model is a representation model, such as BERT, trained for a specific task, like sentiment analysis.
- an embedding model generates general-purpose embeddings that can be used for a variety of tasks not limited to classification, like semantic search

In [122]:
from transformers import pipeline

# Path to our HF model
model_path= "cardiffnlp/twitter-roberta-base-sentiment-latest"

# Load the model: As we load our model, we also load the tokenizer, 
# which is responsible for converting input text into individual tokens
pipe= pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    divice="cuda:0"
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10577.19it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [30]:
for output in pipe(KeyDataset(data["test"], "text")):
    print(type(output))
    print(output)
    break

<class 'dict'>
{'label': 'positive', 'score': 0.9546051621437073}


In [123]:
from transformers.pipelines.pt_utils import KeyDataset
import numpy as np
from tqdm import tqdm
# Run inference - map labels directly since we only get top prediction
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
    # output is {'label': 'negative'/'positive'/'neutral', 'score': ...}
    # Map label to binary: negative=0, positive=1
    if output['label'] == "positive":
        y_pred.append(1)
    elif output['label'] == "negative":
        y_pred.append(0)
    else:  # Handle neutral case if it appears
        # Map neutral based on score or default to negative
        y_pred.append(2)
print(f"Predictions made: {len(y_pred)}")

  0%|          | 0/1066 [00:00<?, ?it/s]

100%|██████████| 1066/1066 [00:15<00:00, 67.33it/s]

Predictions made: 1066


In [124]:
print("Unique labels:", set(output['label'] for output in pipe(KeyDataset(data["test"], "text"))))

Unique labels: {'positive', 'neutral', 'negative'}


In [125]:
len(y_pred)

1066

- tqdm: Provides a progress bar to visualize the loop
- KeyDataset: A utility that efficiently extracts a specific field from a dataset for pipeline processing
- KeyDataset(data["test"], "text"): Extracts only the "text" column from the test dataset (1,066 movie reviews) and feeds it to the pipeline
- pipe(...): Runs the RoBERTa sentiment model on each text. Since return_all_scores=True was set during pipeline creation, it returns scores for all three sentiment classes: negative, neutral, and positive
- np.argmax([negative_score, positive_score]): Returns 0 if negative score is higher, 1 if positive score is higher
- Result: y_pred contains binary predictions (0=negative, 1=positive) for all test samples


<class 'dict'>
{'label': 'positive', 'score': 0.9546051621437073}


In [126]:
from sklearn.metrics import classification_report
filtered_preds = []
filtered_labels = []
def evaluate_performance(y_true, y_pred):
    for pred, label in zip(y_pred, y_true):
        if pred != 2:
            filtered_preds.append(pred)
            filtered_labels.append(label)        
    performance= classification_report(filtered_labels, filtered_preds, target_names=["Negative", "Positive"])
    print(performance)

In [ ]:
print(f"y_true length: {len(filtered_preds)}")
print(f"y_pred length: {len(filtered_labels)}")

y_true length: 786
y_pred length: 786
1066


In [38]:
print("Unique labels:", set(data["test"]["label"]))
# Filter to keep only negative (0) and positive (1)
test_filtered = data["test"].filter(lambda example: example["label"] in [0, 1])
print(f"Original size: {len(data['test'])}")
print(f"Filtered size: {len(test_filtered)}")

Unique labels: {0, 1}
Original size: 1066
Filtered size: 1066


In [127]:
evaluate_performance(data["test"]["label"], y_pred)

              precision    recall  f1-score   support

    Negative       0.81      0.92      0.86       401
    Positive       0.91      0.78      0.84       385

    accuracy                           0.85       786
   macro avg       0.86      0.85      0.85       786
weighted avg       0.86      0.85      0.85       786



In [17]:
print(f"y_true length: {len(data['test']['label'])}")
print(f"y_pred length: {len(y_pred)}")
print(len(data["test"]))

y_true length: 1066
y_pred length: 786
1066


## Classifiaction Tasks that Leverage Embeddings
### Supervised Classification
- Instead of directly using the representation model for classification, we will use an embedding model for generating features, and then those features can be fed into a classifier
- A major benefit of this separation is that we do not need to fine-tune our embedding model, which can be costly. 
- In contrast, we can train a classifier, like a logistic regression, on the CPU instead.

In [48]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# Convert text to embeddings
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)





Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3225.35it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 34/34 [00:01<00:00, 23.48it/s]


the following: This shows that each of our 8,530 input documents has an embedding dimension of 768 and therefore each embedding contains 768 numerical values.

In [49]:
train_embeddings.shape

(8530, 768)

In [50]:
test_embeddings.shape

(1066, 768)

- In the second step, these embeddings serve as the input features to the classifier. The classifier is trainable and not limited to logistic regression and can take on any form as long as it performs classification.

In [51]:
from sklearn.linear_model import LogisticRegression

# Train a logistic Regression model on, our train embedding
clf=LogisticRegression(random_state=42)
clf.fit(train_embeddings, data["train"]["label"])



,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [52]:
# Predict previously unseen instances
y_pred = clf.predict(test_embeddings)
evaluate_performance(data["test"]["label"], y_pred)

              precision    recall  f1-score   support

    Negative       0.85      0.86      0.85       533
    Positive       0.86      0.85      0.85       533

    accuracy                           0.85      1066
   macro avg       0.85      0.85      0.85      1066
weighted avg       0.85      0.85      0.85      1066



- y training a classifier on top of our embeddings, we managed to get an F1 score of 0.85! This demonstrates the possibilities of training a lightweight classifier while keeping the underlying embedding model frozen.

## Tip!

What would happen if we would not use a classifier at all? Instead, we can average the embeddings per class and apply cosine similarity to predict which classes match the documents best:
- train_embeddings: Matrix of shape (num_samples, 768) containing embeddings for training documents
- data["train"]["label"]: Labels (0 or 1) for each training sample
- np.hstack(): Horizontally stacks embeddings with labels, creating a dataframe where columns 0-767 are embedding dimensions and column 768 is the label


In [53]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity

In [55]:
# Average the embeddings of all documents in each target label
df = pd.DataFrame(np.hstack([train_embeddings, np.array(data["train"]["label"]).reshape(-1, 1)]))
averaged_target_embeddings = df.groupby(768).mean().values

# Find the best matching embeddings between evaluation documents and target embeddings
sim_matrix = cosine_similarity(test_embeddings, averaged_target_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

# Evaluate the model
evaluate_performance(data["test"]["label"], y_pred)

              precision    recall  f1-score   support

    Negative       0.85      0.84      0.84       533
    Positive       0.84      0.85      0.84       533

    accuracy                           0.84      1066
   macro avg       0.84      0.84      0.84      1066
weighted avg       0.84      0.84      0.84      1066



averaged_target_embeddings = df.groupby(768).mean().values
- groupby(768): Groups rows by the label column (column 768)
- .mean(): Computes the average of each embedding dimension for each label group
- Result: 2 prototype vectors (one for negative, one for positive), each of dimension 768
- sim_matrix = cosine_similarity(test_embeddings, averaged_target_embeddings) y_pred = np.argmax(sim_matrix, axis=1)
   - cosine_similarity(): Computes similarity between each test embedding and the 2 prototype embeddings
   - Returns a matrix of shape (num_test_samples, 2) where each row contains [similarity_to_negative, similarity_to_positive]
   - np.argmax(axis=1): For each test sample, selects the class with highest similarity (0=negative, 1=positive)

In [56]:
averaged_target_embeddings

array([[ 0.01500115,  0.04630437,  0.00657344, ...,  0.01675619,
         0.01014833, -0.00318564],
       [ 0.00019313,  0.04123216,  0.00064009, ...,  0.02372721,
         0.02076513, -0.0125781 ]], shape=(2, 768))

In [57]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,759,760,761,762,763,764,765,766,767,768
0,0.014930,-0.005548,0.011995,-0.014927,0.006273,-0.003671,-0.026792,0.009564,0.006486,0.019282,...,0.044446,-0.006567,0.004846,0.032993,-0.027030,0.026459,0.005665,-0.012956,0.001537,1.0
1,0.035829,-0.002350,-0.026249,0.025348,-0.011112,0.003088,-0.066249,-0.048875,-0.018407,-0.032934,...,0.029469,-0.034689,0.032908,0.009154,0.029719,0.033257,0.005511,-0.014471,-0.020907,1.0
2,0.040902,0.110522,0.024601,-0.000690,0.005234,0.001776,-0.054121,0.007338,0.000782,0.032753,...,0.002668,-0.029976,-0.031094,-0.004008,0.023225,-0.004077,0.084754,0.016156,0.025994,1.0
3,-0.003141,0.030397,-0.018153,-0.022295,0.021435,0.019211,0.045022,0.083038,0.044163,0.053213,...,0.015134,-0.002314,-0.008651,0.000362,-0.038151,-0.004815,0.003380,0.039602,-0.032613,1.0
4,0.006541,0.044168,0.029882,0.016410,0.003639,0.005672,-0.054883,0.011031,-0.038811,-0.015811,...,-0.032514,-0.028547,-0.006436,0.007190,-0.055309,-0.044386,0.055256,0.098378,-0.002131,1.0


# What If We Do Not Have Labeled Data? (Zero-Shot classification)
- In our previous example, we had labeled data that we could leverage, but this might not always be the case in practice
- Getting labeled data is a resource-intensive task that can require significant human labor.
- To test this, we can perform zero-shot classification, where we have no labeled data to explore whether the task seems feasible.
- Zero-shot classification attempts to predict the labels of input text even though it was not trained on them

In [58]:
# Create embeddding for our labels
label_embeddings = model.encode(["A negative review",  "A positive review"])

In [59]:
from sklearn.metrics.pairwise import cosine_similarity

# Find the best matching label for each document
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)



              precision    recall  f1-score   support

    Negative       0.78      0.77      0.78       533
    Positive       0.77      0.79      0.78       533

    accuracy                           0.78      1066
   macro avg       0.78      0.78      0.78      1066
weighted avg       0.78      0.78      0.78      1066



In [62]:
y_pred

array([1, 1, 1, ..., 0, 1, 0], shape=(1066,))

In [63]:
sim_matrix

array([[ 0.12279505,  0.22788554],
       [ 0.16302872,  0.26313293],
       [ 0.04916785,  0.05250447],
       ...,
       [ 0.00335173, -0.01959533],
       [ 0.12749435,  0.14169864],
       [ 0.12168829,  0.09245002]], shape=(1066, 2), dtype=float32)

## Use more aggressive labels

In [64]:
# Create embeddding for our labels
label_embeddings = model.encode(["A very negative movie review",  "A very positive movie review"])

In [65]:
from sklearn.metrics.pairwise import cosine_similarity

# Find the best matching label for each document
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

In [66]:
evaluate_performance(data["test"]["label"], y_pred)

              precision    recall  f1-score   support

    Negative       0.86      0.73      0.79       533
    Positive       0.76      0.88      0.82       533

    accuracy                           0.80      1066
   macro avg       0.81      0.80      0.80      1066
weighted avg       0.81      0.80      0.80      1066



## Classification Task with generative model
### First approach: Use text-to-text transfer Transformer
Throughout this book, we will explore mostly encoder-only (representation) models like BERT and decoder-only (generative) models like ChatGPT. However, as discussed in Chapter 1, the original Transformer architecture actually consists of an encoder-decoder architecture. Like the decoder-only models, these encoder-decoder models are sequence-to-sequence models and generally fall in the category of generative models.

- An interesting family of models that leverage this architecture is the Text-to-Text Transfer Transformer or T5 model. 

#### Encoder-Decoder Models

In [92]:
# Load the model

pipe = pipeline(
    "text-generation",
    model="google/flan-t5-small",
    device="cuda:0",
    max_length=None,
    max_new_tokens=10,
    do_sample=False
    #return_full_text=False,
)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 4525.04it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTe

In [80]:
data = load_dataset("rotten_tomatoes")

In [73]:
data.column_names

{'train': ['text', 'label'],
 'validation': ['text', 'label'],
 'test': ['text', 'label']}

In [81]:
prompt = "Is the following sentence positive or negative? "
data = data.map(lambda example: {"t5": prompt + example['text']})
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
})

In [75]:
data.column_names

{'train': ['text', 'label', 't5'],
 'validation': ['text', 'label', 't5'],
 'test': ['text', 'label', 't5']}

In [87]:
# Run inference
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "t5")), total=len(data["test"])):
    text = output[0]["generated_text"]
    #print(text)
    y_pred.append(0 if text == "negative" else 1)


100%|██████████| 1066/1066 [00:49<00:00, 21.48it/s]


In [86]:
evaluate_performance(data["test"]["label"], y_pred)

              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00       533
    Positive       0.50      1.00      0.67       533

    accuracy                           0.50      1066
   macro avg       0.25      0.50      0.33      1066
weighted avg       0.25      0.50      0.33      1066



d:\2026-courses\LLMs-Handson\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\2026-courses\LLMs-Handson\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\2026-courses\LLMs-Handson\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


#### Version2: Change the prompt

In [93]:
prompt = " You are a movie critic expert that can classify movie reviews as positive or negative. Is the following sentence positive or negative? Please rteurn as output 0 or 1 as 0 is negative 1 positive"
data = data.map(lambda example: {"t6": prompt + example['text']})
data

Map: 100%|██████████| 1066/1066 [00:00<00:00, 16648.09 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 't5', 't6'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 't5', 't6'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 't5', 't6'],
        num_rows: 1066
    })
})

In [ ]:
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "t6")), total=len(data["test"])):
    text = output[0]["generated_text"]
    print(text)
    y_pred.append(0 if text == "negative" else 1)

In [109]:
from transformers import pipeline
import numpy as np
from tqdm import tqdm

# Create the pipeline
pipe = pipeline(
    "text-classification",  # Use text2text-generation for T5 models
    model="google/flan-t5-small",
    device="cuda:0"
)

# Better prompt format for Flan-T5
def create_prompt(text):
    # Simpler, more direct prompt
    return f"Classify this movie review as positive or negative. Answer with only 'positive' or 'negative'.\n\nReview: {text}\n\nClassification:"

# Run inference
y_pred = []
for i in tqdm(range(len(data["test"]))):
    text = data["test"]["text"][i]
    prompt = create_prompt(text)
    
    # Generate with specific parameters
    output = pipe(
        prompt,
        max_new_tokens=5,        # Only need 1-2 tokens for answer
        do_sample=False,          # Deterministic output
        return_full_text=False    # IMPORTANT: Don't return input
    )
    
    # Extract the generated text
    #generated = output[0]['generated_text'].strip().lower()
    
    # Map to binary labels
    if output[0]['label'] == 'LABEL_1':
        y_pred.append(1)
    elif output[0]['label'] == 'LABEL_0':
        y_pred.append(0)
    else:
        # Default to negative if unclear
        print(f"Unclear response: '{output}'")
        y_pred.append(0)

print(f"Predictions made: {len(y_pred)}")

Loading weights: 100%|██████████| 189/189 [00:00<00:00, 4973.67it/s]
T5ForSequenceClassification LOAD REPORT from: google/flan-t5-small
Key                                 | Status     | 
------------------------------------+------------+-
lm_head.weight                      | UNEXPECTED | 
classification_head.dense.weight    | MISSING    | 
classification_head.out_proj.weight | MISSING    | 
classification_head.out_proj.bias   | MISSING    | 
classification_head.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 1066/1066 [00:29<00:00, 36.36it/s]

Predictions made: 1066


In [110]:
evaluate_performance(data["test"]["label"], y_pred)


              precision    recall  f1-score   support

    Negative       0.50      1.00      0.67       533
    Positive       0.00      0.00      0.00       533

    accuracy                           0.50      1066
   macro avg       0.25      0.50      0.33      1066
weighted avg       0.25      0.50      0.33      1066



d:\2026-courses\LLMs-Handson\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\2026-courses\LLMs-Handson\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\2026-courses\LLMs-Handson\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [108]:
pipe = pipeline(
    "text-classification",  # Use text2text-generation for T5 models
    model="google/flan-t5-small",
    device="cuda:0"
)

output = pipe(
        prompt,
        max_new_tokens=5,        # Only need 1-2 tokens for answer
        do_sample=False,          # Deterministic output
        return_full_text=False    # IMPORTANT: Don't return input
    )

output

Loading weights: 100%|██████████| 189/189 [00:00<00:00, 4725.00it/s]
T5ForSequenceClassification LOAD REPORT from: google/flan-t5-small
Key                                 | Status     | 
------------------------------------+------------+-
lm_head.weight                      | UNEXPECTED | 
classification_head.dense.weight    | MISSING    | 
classification_head.out_proj.weight | MISSING    | 
classification_head.out_proj.bias   | MISSING    | 
classification_head.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[{'label': 'LABEL_0', 'score': 0.6946144104003906}]

## Use OpenAI: ChatGPT for Classification

In [ ]:
# pip install openai

In [ ]:
import openai
Client = openai.OpenAI(api_key="sk-xxxxx")

In [118]:
def chatgpt_generation(prompt, document, model="gpt-3.5-turbo-0125"):
    """Generate an output based on a prompt and an input document"""
    messages =[
        {
            "role": "system",
            "content":"You are a hlpful assistant."
        },
        {
            "role":"user",
            "content": prompt.replace("[DOCUMENT]", document)
        }
    ]

    chat_completion = Client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.0
    )

    return chat_completion.choices[0].message.content
    

### Test

In [ ]:
# Define a prompt template as a base
prompt = """Predict whether the following document is a positive or negative movie review:

[DOCUMENT]

If it is positive return 1 and if it is negative return 0. Do not give any other answers.
"""

# Predict the target using GPT
document = "unpretentious , charming , quirky , original"
chatgpt_generation(prompt, document)

The next step would be to run one of OpenAI's model against the entire evaluation dataset. However, only run this when you have sufficient tokens as this will call the API for the entire test dataset (1066 records).

In [ ]:
# You can skip this if you want to save your (free) credits
predictions = [chatgpt_generation(prompt, doc) for doc in tqdm(data["test"]["text"])]

In [ ]:
# Extract predictions
y_pred = [int(pred) for pred in predictions]

# Evaluate performance
evaluate_performance(data["test"]["label"], y_pred)



## Tip
- When dealing with external API, you might run into rate limit errors. This appear when we call the API too often as some APIs might limit the rate error with which you can use it per minute or hour.
- To prevent these errors, we can implement several methods for retrying the request, including something referred to as exponential backoff.
  - It performs a short sleep each time we hit a rate limit error and then retries the unsuccessful request.
  - Whenever it is unsuccessful again, the sleep length is increased until the request is successful or we hit a maximum number of retries.

# Summary: Movie Classification Problem
- Challenge: Use pre-trained sentiment models on datasets with different label schema
  - In our situation:
    - Dataset: Rotten Tomatoes dataset ==> Labels: positive / negative (binary)
    - Model: cardiffnlp/twitter-roberta-base-sentiment-latest→ Outputs: positive / neutral / negative (3 classes)
the core issue: We are trying to map 3-class model -> 2-class dataset. Question: What should we do with the neutral class?
## Option1: Ignore Neutral for best evaluation
- Only keep predictions where model predicts positive and negative
- Drop neutral predicitions during evaliation
- Pros:
  - Clean evaluation
  - No artificial bias
- Cons
  - Lose some samples

## Option2: Map Neutral to Negative (Common Practical choice): 
- positive → positive
- neutral → negative
- negative → negative

Why? - In many sentiment tasks, neutral ≈ not positive
Pros: 
  - Keeps all samples
  - Simple
cons:
 - Slight bias towrad negative class

## Option3: Map Neutral → Positive (Less Common)
Only valid if:
 - Your dataset definition treats neutral as slightly positive. Usually not recommended for Rotten Tomatoes

## Option4: Threshold-Based Strategy (Best Advanced Approach)
- Instead of using argmax, use probabilities:

if P(positive) > 0.6:
    label = "positive"
else:
    label = "negative"
Neutral gets absorbed based on confidence
- Pros:
 - More Robust
 - Uses model confidence
- Cons:
 - Requires tuning

## Option5: Fine-tune Model (Best Long-Term solution)
- Fine-tune:
- RoBERTa
on:
 - Rotten Tomatoes dataset

to output: Only 2 labels
- Pros:
  - Best performance
  - Clean alignment
- Cons:
  - Requires training